# 0. Enviornment Setup

In [17]:
# Setup Uni-Mol 
!git clone https://github.com/deepmodeling/Uni-Mol.git

Cloning into 'Uni-Mol'...
remote: Enumerating objects: 1504, done.
remote: Counting objects: 100% (640/640), done.
remote: Compressing objects: 100% (288/288), done.
remote: Total 1504 (delta 490), reused 364 (delta 352), pack-reused 864 (from 1)
Receiving objects: 100% (1504/1504), 23.67 MiB | 3.20 MiB/s, done.
Resolving deltas: 100% (919/919), done.


In [70]:
# Install dependancies
!pip uninstall rdkit -y
!pip install rdkit==2023.9.5

Found existing installation: rdkit 2023.9.5
Uninstalling rdkit-2023.9.5:
  Successfully uninstalled rdkit-2023.9.5
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
     |████████████████████████████████| 34.4 MB 6.2 MB/s eta 0:00:01


In [20]:
# Install dependancies
%%bash
pip install lmdb
pip install torch

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [26]:
!pip install protobuf==<3.20.3>
!pip install typing-extensions==<4.6.0>

/bin/bash: -c: line 0: syntax error near unexpected token `newline'
/bin/bash: -c: line 0: `pip install protobuf==<3.20.3>'
/bin/bash: -c: line 0: syntax error near unexpected token `newline'
/bin/bash: -c: line 0: `pip install typing-extensions==<4.6.0>'


In [27]:
# Install  Uni-Core
!git clone https://github.com/dptech-corp/Uni-Core
!cd Uni-Core && pip install -e .

fatal: destination path 'Uni-Core' already exists and is not an empty directory.
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Obtaining file:///Uni-Core
  Attempting uninstall: unicore
    Found existing installation: unicore 0.0.1
    Uninstalling unicore-0.0.1:
      Successfully uninstalled unicore-0.0.1
  Running setup.py develop for unicore


In [50]:
# Import Packages
import  os, sys, time, pickle, lmdb

import  pandas            as pd
import  numpy             as np
import  lightgbm          as lgb
import  matplotlib.pyplot as plt 
from    tqdm                          import tqdm
from    rdkit                         import Chem
from    rdkit.Chem                    import AllChem
from    rdkit.Chem.rdMolDescriptors   import GetMorganFingerprintAsBitVect

from    sklearn.linear_model          import RidgeClassifier, LogisticRegression
from    sklearn.svm                   import LinearSVC
from    sklearn.ensemble              import ExtraTreesClassifier
from    sklearn.metrics               import roc_auc_score, fbeta_score

import  torch
import  torch.nn as nn
from    torch.utils.data               import Dataset, DataLoader

In [29]:
!pwd
!ls

/
Uni-Core  dev		 lib	 nvidia    run	       submission.csv
Uni-Mol   etc		 lib32	 opt	   sbin        sys
bin	  exec_time.txt  lib64	 personal  share       tmp
bohr	  home		 libx32  proc	   speedUp.sh  usr
boot	  jupyter.log	 media	 rapids    srv	       var
data	  launch.py	 mnt	 root	   start.log   workspace


In [35]:
# Import Model
sys.path.append("Uni-Mol")
sys.path.append("Uni-Core")
import unicore
import unimol2
import unimol
from   unimol.models import unimol
#from   unimol2.models import unimol2

# 1. Load Data

In [37]:
# Mount bohr CNS Database
DIR_PATH = '/bohr/ai4scup-cns-5zkz/v3/'

# Divide DataSet
df_data          = pd.read_csv(os.path.join(DIR_PATH, 'mol_train.csv'))
df_train, df_val = df_data[:500], df_data[500:]
df_test          = pd.read_csv(os.path.join(DIR_PATH, 'mol_test.csv'))

# Check quantity
df_train.shape, df_val.shape, df_test.shape

((500, 2), (200, 2), (367, 2))

In [38]:
# Def. helper funct.
def fe_fingerprint(smi_list, name='morgan'):                    # 定义fe_fingerprint函数，输入为smi_list和name，默认name为'morgan'
    fp_list = []                                                # 初始化一个空列表fp_list
    for smi in smi_list:                                        # 遍历输入列表smi_list
        mol = Chem.MolFromSmiles(smi)                           # 将输入列表中的smi转换为mol格式
        fp = GetMorganFingerprintAsBitVect(mol, 5, nBits=1024)  # 计算mol的Morgan指纹，并将其转换为位向量
        fp = fp.ToBitString()                                   # 将位向量转换为比特字符串
        fp_list.append([int(item) for item in list(fp)])        # 将比特字符串转换为整数列表，并添加到fp_list
    return fp_list                                              # 返回fp_list

def sigmoid(x):                                                 # 定义sigmoid函数，输入为x
    return 1/(np.exp(-x)+1)                                     # 计算并返回sigmoid函数值

In [39]:
X_fp = np.array(fe_fingerprint(df_data['SMILES']))      # 使用fe_fingerprint函数将数据集中的SMILES表示转换为Morgan指纹，并将其转换为NumPy数组
X_train_fp, X_val_fp = X_fp[:500], X_fp[500:]           # 将Morgan指纹数组分为训练集（前500个样本）和验证集（剩余样本）
X_test_fp = np.array(fe_fingerprint(df_test['SMILES'])) # 使用fe_fingerprint函数将测试集中的SMILES表示转换为Morgan指纹，并将其转换为NumPy数组

In [40]:
X_fp.shape, X_train_fp.shape, X_val_fp.shape, X_test_fp.shape

((700, 1024), (500, 1024), (200, 1024), (367, 1024))

## 2. Train & Pred. with Diff. ML model

本次赛题采用 `F2-score` 作为评价指标。
`F2-score` 是 `F1-beta` 取 $beta = 2$ 时的特殊形式。  `F1-beta` 允许我们调整精确率（Precision）和召回率（Recall）之间的权重，其一般形式如下：

$$F1-beta = (1 + beta^2) * (Precision * Recall) / (beta^2 * Precision + Recall)$$

在本次赛事中，我们更在意模型的召回率，取 $beta = 2$。此时我们得到 `F2-score`，`F2-score` 的计算公式为：

$$F2-score = 5 * (Precision * Recall) / (4 * Precision + Recall)$$

其中，Precision = 真正例（TP）/（真正例（TP）+假正例（FP）），Recall = 真正例（TP）/（真正例（TP）+假负例（FN））。
TP（True Positive）表示实际为正例且预测为正例的样本数；TN（True Negative）表示实际为负例且预测为负例的样本数；FP（False Positive）表示实际为负例但预测为正例的样本数；FN（False Negative）表示实际为正例但预测为负例的样本数。

最终排行榜展示结果只保留 `F2-score` 的四位小数。

## 2.1. LightGBM 模型：

In [41]:
# 将数据特征矩阵转换为 LightGBM 指定格式，(特征向量，对应标签)
lgb_train = lgb.Dataset(X_train_fp, label=df_train["TARGET"])
lgb_valid = lgb.Dataset(X_val_fp, label=df_val["TARGET"])
# 设定 LightGBM 训练参，查阅参数意义：https://lightgbm.readthedocs.io/en/latest/Parameters.html
lgb_params = {
    "objective": "binary",  # 指定任务类别为二分类
    "seed": hash("AI4S-Cup") % 2023,  # 设定随机种子
    "verbose": -1,  # 禁用输出（可选）
}

# 训练模型，参数依次为：导入模型设定参数、导入训练集、设定模型迭代次数（100）、导入验证集
model = lgb.train(lgb_params, lgb_train, num_boost_round=100, valid_sets=lgb_valid)
# 用验证集进行模型预测（选择训练中最好的一次）
valid_pred = model.predict(X_val_fp, num_iteration=model.best_iteration)
# 生成预测标签结果，如果概率大于阈值则为 1，否则为 0
threshold = 0.5
valid_result = [1 if x > threshold else 0 for x in valid_pred]
# 计算验证集 F2 Score 分数
valid_score = fbeta_score(df_val["TARGET"], valid_result, beta=2)
print(f"Valid Score: {valid_score}")

Valid Score: 0.688622754491018


## 2.2. Extra Trees（极端随机树）分类器模型：

In [42]:
et = ExtraTreesClassifier(n_estimators=100, 
                          criterion='entropy',
                          max_depth=5, 
                          min_samples_split=5, 
                          random_state=42,
                          n_jobs=-1).fit(X_train_fp,df_train['TARGET'])
y_predict_et = et.predict(X_val_fp)
valid_score = fbeta_score(df_val['TARGET'],y_predict_et,beta=2)
print(f"Valid Score: {valid_score}")

Valid Score: 0.4


## 2.3. 逻辑回归模型

In [43]:
lr = LogisticRegression(C=0.01).fit(X_train_fp,df_train['TARGET'])
y_predict_lr = lr.predict(X_val_fp)
valid_score = fbeta_score(df_val['TARGET'],y_predict_lr, beta=2)
print(f"Valid Score: {valid_score}")

Valid Score: 0.5128205128205129


## 2.4. 线性支持向量分类模型

In [44]:
svc = LinearSVC(C=0.01,max_iter=1000,random_state=42).fit(X_train_fp,df_train['TARGET'])
y_predict_svc = svc.predict(X_val_fp)
valid_score = fbeta_score(df_val['TARGET'],y_predict_svc, beta=2)
print(f"Valid Score: {valid_score}")

Valid Score: 0.6626506024096386


---

根据F2-score来评估模型的好坏：

1. 接近1的F2-score表示模型性能很好，意味着模型在正确预测正例方面做得很好，同时尽量减少了假负例（即避免将实际为正的样本预测为负）。

2. 接近0的F2-score表示模型性能很差，意味着模型在正确预测正例方面做得很差，同时产生了很多假负例。

# 3. Train & Pred. with Uni-Mol

# 4. 预测&生成结果文件

本次赛题要求notebook能按照指定格式生成预测结果csv文件，并且存在指定的路径。

In [45]:
svc = LinearSVC(C=0.01,
                max_iter=1000,
                random_state=42).fit(X_fp,df_data['TARGET'])
y_predict_svc = svc.predict(X_test_fp)

In [46]:
sub = pd.read_csv(os.path.join(DIR_PATH, 'mol_sample_submission.csv'))[['SMILES']]
sub['TARGET'] = y_predict_svc
sub.to_csv('./submission.csv', index=False, header=True)